# boring — CUDA tensor numerical check on Colab

Validates the CUDA tensor paths on real hardware against a CPU reference: dynamic `kernel.tensors.linear` (cuBLAS fp32, fused bias), a device `matmulTile`, and a Q8_0 quantized prefill. Runs three configurations: exact fp32 (default), opt-in TF32 (`[tensor.matmul] algorithm = "tf32"`, sm_80+), and opt-in half prefill (`[tensor.linear.prefill] precision = "half"`, WMMA f16, sm_70+).

**Runtime**: T4 (sm_75) covers fp32 and half; TF32 needs A100/L4.

Expected: fp32 errors ≈ 1e-6 (quantized prefill ≈ 1e-4); TF32 ≈ 1e-3..1e-2; half prefill ≈ 1e-2 on the quantized check (clearly above fp32's ≈ 2e-6). On a T4 the TF32 cell is skipped automatically.

In [ ]:
# Allocated GPU + CUDA driver version
!nvidia-smi

In [ ]:
# nvcc is not always on PATH on Colab: look in /usr/local/cuda*, install the toolkit as a last resort
import glob, os, shutil, subprocess
if not shutil.which("nvcc"):
    found = sorted(glob.glob("/usr/local/cuda*/bin/nvcc"))
    if found:
        os.environ["PATH"] = os.path.dirname(found[-1]) + ":" + os.environ["PATH"]
if not shutil.which("nvcc"):
    print("nvcc not found: installing the CUDA toolkit (a few minutes)...")
    subprocess.run("apt-get update -qq && apt-get install -y -qq nvidia-cuda-toolkit", shell=True, check=True)
assert shutil.which("nvcc"), "nvcc still missing: check that the runtime has a GPU (Runtime > Change runtime type)"
print(shutil.which("nvcc"))
!nvcc --version


In [ ]:
# Rust is NOT preinstalled on Colab, unlike CUDA
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable
import os
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
!rustc --version && cargo --version

## Clone the repo

The project is cloned from GitHub (`github.com/mlanoe/boring`). Change `BRANCH` to the branch you want to test (push it first).

In [ ]:
BRANCH = "main"  # <- replace with your working branch
!git clone --branch $BRANCH --depth 1 https://github.com/mlanoe/boring.git
%cd boring

In [ ]:
# Build the boring compiler itself (a Rust CLI, nothing GPU-specific here)
!cargo build --release --quiet
BORING = "./target/release/boring"
!$BORING --help | head -5

In [ ]:
import pathlib, re, subprocess
PROJECT = pathlib.Path('examples/cuda_tensor_check')
SOURCE = PROJECT / 'main.br'
out = subprocess.run(['nvcc','--version'],capture_output=True,text=True).stdout
m = re.search(r'release (\d+)\.(\d+)', out)
FEATURE = f'cuda-{m.group(1)}{int(m.group(2)):02d}0'
print('cudarc feature:', FEATURE)

def run(label, toml):
    (PROJECT / 'boring.toml').write_text(toml)
    subprocess.run([BORING, 'build', '--target', 'cuda', str(SOURCE)], check=True)
    d = PROJECT / 'main_cuda'
    ct = d / 'Cargo.toml'
    ct.write_text(re.sub(r'(cudarc\s*=\s*\{[^}]*features\s*=\s*\[)([^\]]*)(\])', lambda mo: f'{mo.group(1)}"driver", "nvrtc", "cublas", "{FEATURE}"{mo.group(3)}', ct.read_text()))
    r = subprocess.run('cargo run --release --quiet', shell=True, cwd=d, capture_output=True, text=True)
    print(f'== {label} ==');print(r.stdout);print(r.stderr[-2000:])


## fp32 (default) — errors must be ≤ 1e-3

In [ ]:
run('fp32', '[package]\nname = "cuda_tensor_check"\n')

## TF32 opt-in (sm_80+ only) — errors expected ≈ 1e-3..1e-2

In [ ]:
cap = float(subprocess.run(['nvidia-smi','--query-gpu=compute_cap','--format=csv,noheader'],capture_output=True,text=True).stdout.split()[0])
if cap >= 8.0:
    run('tf32', '[package]\nname = "cuda_tensor_check"\n\n[tensor.matmul]\nalgorithm = "tf32"\n')
else:
    print(f'Skipped: TF32 needs sm_80+, this GPU is sm_{int(cap*10)} (use an A100/L4 runtime).')


## Half prefill opt-in (sm_70+) — `linear` and `tile` stay fp32-exact; the quantized prefill error must be clearly above the fp32 run (≈1e-2): if it equals the fp32 value, the half kernel did not run

In [ ]:
run('half', '[package]\nname = "cuda_tensor_check"\n\n[tensor.linear.prefill]\nprecision = "half"\n')